# Proyecto 1 — Analítica de Datos
## Modelamiento de la precipitación semanal en el Valle del Cauca

**Universidad Autónoma de Occidente** · Ingeniería de Datos e Inteligencia Artificial  
Analítica de Datos 2026-2S · Prof. Johann A. Ospina  
César Armando Reyes Oliveros — 2236379

---

### Para qué sirve este cuaderno

Es el **cuaderno de trabajo**: registra qué hacemos, por qué lo hacemos, y en qué
material de clase se apoya cada decisión. Sirve para que el grupo siga el hilo y
para preparar la sustentación.

> ⚠️ **Esto NO es el entregable.** El enunciado exige un **PDF de máximo 10 páginas**
> más **un único archivo `.R`**, y prohíbe explícitamente entregar en R Markdown o
> solo en Colab. El código canónico vive en **`R/proyecto1.R`**; aquí solo se explica.

### Cómo está organizado

Cada paso responde cuatro preguntas:

| | |
|---|---|
| **Qué** | la operación concreta |
| **Por qué** | qué problema resuelve, y qué pasa si no se hace |
| **Material** | el script o documento del curso que lo respalda |
| **Resultado** | los números que salieron |


---
## Paso 0 — El encargo y los datos

### Qué pide el enunciado

Modelar la precipitación semanal en el Valle del Cauca en función de covariables
ambientales, y **evaluar el papel de la correlación espacial**. Cubrir cuatro bloques:
EDA → Modelación → Validación → Predicción, justificando cada uno con desarrollo
teórico paso a paso.

La restricción que manda sobre todo lo demás:

> *"Utilizar únicamente los scripts, procedimientos y metodologías trabajados en clase.
> Librerías adicionales permitidas solo para: elaboración de gráficos y carga de mapas
> o archivos de Excel."*

Esa lista no está escrita en ninguna parte. Hay que deducirla de los scripts del
profesor, que están en **`material_clase/`**.

### El molde

**`material_clase/Practice Geostatistics/Ejemplo4_Geoestadística.R`** es el molde de
este proyecto: mismo departamento, mismos rasters, mismo flujo de trabajo.

| Sección de Ejemplo4 | Qué hace |
|---|---|
| Cargar imágenes | apila los `.tif` con `rast()` |
| Borde del valle | `as.polygons()` sobre la máscara de datos válidos |
| Puntos de muestreo | `spatSample(..., method = "random")` |
| Proyección | grados → km con la fórmula del coseno de la latitud |
| EDA espacial | histogramas, mapa de posting, dispersión vs coordenadas |
| Extraer tendencia | `lm(z ~ x_km + y_km)` |
| Distancias | `dist()` |
| Semivariograma | nube de pares + promedio por bins con `tapply()` |
| Ajuste de modelo | `optim()` con L-BFGS-B, multi-arranque |
| Predicción | sistema kriging resuelto con `solve()` |
| Validación cruzada | LOOCV a mano |

Nuestro script sigue ese orden exacto.

### Librerías permitidas

✅ Con precedente de clase: `terra`, `sp`, `gstat`, `geodata`, `chirps`, `ggplot2`,
`gridExtra`, `geostan`, `openxlsx`, `dplyr`, y todo base R (`lm`, `optim`, `solve`,
`dist`, `tapply`, `cut`).

❌ Sin precedente: `automap`, `spdep`, `sf`, `caret`, `randomForest`, `mgcv`, `tmap`,
`leaflet`, el paquete viejo `raster`, y `terra::interpIDW`.

**Decisión:** usamos **solo `terra` + base R**. Es el subconjunto más defendible y es
exactamente lo que usa el molde.

### Verificación de los datos

Antes de tocar nada se comprobó que el dataset es el original del profesor:

```
md5  d1bd258c058004a7877617817027b60a   datos_proyecto_1.zip   respaldo == repo ✅
md5  813557728ea4ceee9b5977e37bc68789   enunciado PDF          idéntico      ✅
CRC  229 de 229 archivos .tif extraídos coinciden con el zip                 ✅
```


---
# Paso 1 — Auditoría de los datos

**Qué.** Cargar las cuatro capas y verificar geometría, valores centinela y cobertura
real antes de modelar.

**Por qué.** El enunciado afirma que las capas son *"directamente comparables píxel a
píxel"*. Es cierto en geometría y **falso en cobertura**. Si se toma la afirmación
como buena y se mete todo a un `lm()`, el modelo queda apoyado en datos inventados.

**Material.** `Ejemplo4_Geoestadística.R` (carga con `rast`), `carpeta/Script_Class5.R`
(manejo de rasters con `terra`: `rast`, `values`, `crs`).


In [ ]:
library(terra)
set.seed(2026)

DIR        <- "data/datos_proyecto_1/imagenes_semanales"
DIR_SALIDA <- "resultados"

r_precip <- rast(file.path(DIR, "chirps_climatologia_semanal_valle.tif"))
r_temp   <- rast(file.path(DIR, "power_temp_climatologia_semanal_valle.tif"))
r_rad    <- rast(file.path(DIR, "power_radiacion_climatologia_semanal_valle.tif"))
r_alt    <- rast(file.path(DIR, "altitud_valle.tif"))


## 1.1 Geometría

**Resultado:** las cuatro capas comparten grilla exacta.

```
              filas cols bandas  res   xmin  xmax  ymin  ymax
Precipitacion    39   37     52 0.05 -77.55 -75.7  3.05  5.00
Temperatura      39   37     52 0.05 -77.55 -75.7  3.05  5.00
Radiacion        39   37     52 0.05 -77.55 -75.7  3.05  5.00
Altitud          39   37      1 0.05 -77.55 -75.7  3.05  5.00
```

39 × 37 = **1 443 píxeles** por banda, de 0,05° (≈ 5,6 km), en EPSG:4326.
Las 52 bandas son las semanas ISO del año.


## 1.2 Valores centinela

**Qué.** CHIRPS codifica "sin dato" como un número negativo enorme, no como `NA`.

**Por qué importa.** La lluvia no puede ser negativa. Si no se filtra, la media del
departamento sale en decenas de miles de milímetros negativos y **todo lo que sigue
es basura**: la tendencia, el variograma, el kriging.

**Resultado:**

```
Mínimo crudo observado:  -76867.3
Celdas negativas:            104
Mínimo tras limpiar:        2.77 mm   ← ya es un valor físico
```


In [ ]:
# La precipitación es una cantidad no negativa: todo valor < 0 es centinela.
r_precip[r_precip < 0] <- NA


## 1.3 La máscara del área de estudio

**Qué.** Definir qué celdas son "el Valle del Cauca".

**Por qué.** El rectángulo de 1 443 píxeles **no** es el departamento: incluye océano
Pacífico y territorio vecino. Sin una máscara no hay denominador honesto para medir
nada.

> 🔴 **El error que cometimos.** La primera versión midió cobertura contra las 1 443
> celdas del rectángulo. Resultado: `Altitud → 47,7 % → SE DESCARTA`. Descartaba la
> mejor covariable. Mismo dato, dos denominadores, conclusiones opuestas:
> 
> ```
> altitud / rectángulo   = 688/1443 = 47,7 %  →  "media cobertura"
> altitud / departamento =  688/688 =  100 %  →  "cobertura total"
> ```
> 
> **Define la máscara antes de calcular cualquier porcentaje.**


In [ ]:
SEMANA_DIAG <- 42

mascara <- !is.na(r_alt) & !is.na(r_precip[[SEMANA_DIAG]])
N_VALLE <- sum(values(mascara), na.rm = TRUE)

cat(sprintf("Celdas del rectángulo:      %d\n", ncell(r_alt)))
cat(sprintf("Celdas del área de estudio: %d\n", N_VALLE))


## 1.4 Cobertura efectiva — el hallazgo que cambia el modelo

**Qué.** Contar, dentro del área de estudio, cuántas celdas tiene realmente cada
variable y cuántos valores distintos toma.

**Resultado:**

| Variable | Celdas | % del área | Valores distintos | |
|---|---|---|---|---|
| Precipitación | 688 | 100,0 % | 688 | ✅ |
| Altitud | 688 | 100,0 % | 688 | ✅ |
| Temperatura | 123 | 17,9 % | 56 | ❌ |
| **Radiación** | **28** | **4,1 %** | **3** | ❌ |

**Por qué pasa.** CHIRPS es nativo 0,05°; NASA POWER es nativo **0,5°** — diez veces
más grueso. Al llevar POWER a la grilla fina solo sobrevivieron los centros originales;
el resto quedó `NA`. El enunciado dice "remuestreadas a la resolución de CHIRPS", pero
eso fue reproyección de rejilla, **no relleno**.

**Por qué importa.** La radiación, dentro del Valle, son **3 números**. No es un
gradiente continuo: es un escalón que separa costa / valle / cordillera. Meterla a un
`lm()` como variable continua le atribuye a "radiación" lo que en realidad es
**posición geográfica** — que ya entra al modelo como `x_km` e `y_km`.

**Decisión.** Temperatura y radiación **quedan fuera**. El modelo se construye con
**altitud + coordenadas**, las tres con 100 % de cobertura y cero dato inventado.

Esto además evita `terra::interpIDW`, que es como se rellenarían esos huecos y que
**no tiene precedente en ningún script de clase** (verificado con `grep` sobre todo
`material_clase/`).


## 1.5 Elección de la semana de estudio

**Qué.** Elegir la semana a modelar calculando el ciclo anual, no suponiéndola.

**Por qué.** "Octubre es lluvioso" es una creencia razonable, pero el dato decide.
Justificar la elección con evidencia es exactamente lo que pide el enunciado.

**Resultado:**

```
Semana más lluviosa (climatología 2010-2025): 44 (91,9 mm)
Semana más seca:                               3 (28,7 mm)
```

La semana **44** (principios de noviembre), no la 42. Se ve el régimen **bimodal**
andino: dos picos al año, abril-mayo y octubre-noviembre.


In [ ]:
media_semanal <- sapply(1:nlyr(r_precip),
                        function(k) mean(values(r_precip[[k]]), na.rm = TRUE))

SEMANA <- which.max(media_semanal)   # -> 44

plot(1:52, media_semanal, type = "o", pch = 19, col = "#08519c",
     xlab = "Semana ISO", ylab = "Precipitación media (mm)",
     main = "Ciclo anual - Valle del Cauca (CHIRPS 2010-2025)")
abline(v = SEMANA, col = "red", lty = 2, lwd = 2)


![Ciclo anual](resultados/fig01_ciclo_anual.png)

**Interpretación.** Los dos máximos corresponden al paso de la Zona de Convergencia
Intertropical sobre la región, dos veces al año. El mínimo de la semana 3 es el
veranillo de enero. El rango va de 28,7 a 91,9 mm — la semana elegida tiene más del
triple de lluvia que la más seca, así que la señal espacial va a ser fuerte.


---
# Paso 2 — Puntos de muestreo

## 2.1 El borde real del departamento

**Qué.** Convertir la máscara de celdas válidas en un polígono.

**Por qué.** Los puntos de muestreo hay que sortearlos **dentro** del departamento.
Si se sortean sobre el rectángulo, caen en el Pacífico.

**Material.** `Ejemplo4_Geoestadística.R`, líneas 21-24 — es literalmente este patrón.


In [ ]:
borde <- as.polygons(mascara, dissolve = TRUE)
borde <- borde[borde[[1]] == 1, ]      # <- la línea que la gente olvida

# CHIRPS (787 celdas) desborda el departamento (688) y cubre mar abierto.
# Sin recortar, el mapa pinta lluvia sobre el Pacífico.
precip <- mask(r_precip[[SEMANA]], mascara, maskvalues = c(FALSE, NA))

plot(precip, main = "Precipitación semana 44 (mm) - Valle del Cauca")
plot(borde, add = TRUE, border = "black", lwd = 1.5)


### La línea que la gente olvida

`as.polygons()` sobre un raster lógico devuelve **dos** polígonos: el de los `TRUE`
(atributo 1) y el de los `FALSE` (atributo 0). Sin `borde[borde[[1]] == 1, ]` el
"borde" incluye el océano.

### Verificación

```
Polígonos devueltos por as.polygons(): 2
Área del borde conservado:        21 125 km²
```

El Valle del Cauca real tiene **22 140 km²**. Coincidencia del 95 % → la máscara sí
es el departamento, no un recorte arbitrario.

### El recorte de CHIRPS

```
Celdas de CHIRPS antes del recorte:  787
Celdas tras recortar al área:        688
```

CHIRPS traía 99 celdas sobre mar abierto, donde SRTM no tiene altitud. La primera
versión de esta figura pintaba lluvia sobre el Pacífico.


![Borde del Valle](resultados/fig02_borde_valle.png)

**Interpretación — y es el hallazgo central del EDA.**

El gradiente **Oeste → Este** es enorme y limpio:

| Zona | Lluvia semana 44 |
|---|---|
| Vertiente pacífica (Buenaventura, verde) | **200+ mm** |
| Valle interandino (blanco) | **30-50 mm** |

Un factor de **5 a 7 veces** en ~150 km. Es el efecto orográfico: la humedad del
Pacífico choca contra la cordillera Occidental, descarga en la vertiente y llega seca
al valle del río Cauca.

**Consecuencia para el modelo:** buena parte de la variación de la precipitación es
**tendencia de gran escala en la longitud**, no estructura aleatoria. Por eso el
modelo se separa en dos partes —

1. una **tendencia** determinística, `lm(precip ~ altitud + x_km + y_km)`, y
2. un **residuo espacialmente correlacionado**, que es lo que interpola el kriging.

Los dos huecos blancos dentro del borde son celdas sin altitud; `dissolve = TRUE` los
respetó como agujeros del polígono, que es el comportamiento correcto.


---
# Lo que sigue

| Paso | Qué | Bloque del enunciado |
|---|---|---|
| 2.2 | Sorteo de puntos con `spatSample` y proyección a km | — |
| 3 | EDA + autocorrelación: Moran's I y Geary's C a mano | **1. EDA** |
| 4 | Tendencia con `lm()` e interpretación de coeficientes | **2. Modelación** |
| 5 | Semivariograma: nube → bins → ajuste con `optim()` | **2. Modelación** |
| 6 | Sistema kriging con `solve()` | **2. Modelación** |
| 7 | LOOCV y lectura honesta del R² | **3. Validación** |
| 8 | Mapa de predicción + mapa de incertidumbre | **4. Predicción** |

### Recordatorio de entrega

| | |
|---|---|
| **Plazo** | domingo **13-sep-2026, 23:59** — solo Moodle |
| **Documento** | PDF, máx. **10 páginas**, una sola columna |
| **Código** | **un único** `.R`, reproducible, en archivo aparte |
| **Prohibido** | R Markdown, solo-Colab, manuscrito |
| **Si se usó IA** | `.docx` aparte con los prompts y la herramienta |
| **Sustentación** | se sortea un integrante; ausencia sin excusa = **0.0** |
